# Maximum rainfall: Gumbel fit (fictitious data)
This notebook does ALL the calculation. The document `report.pltx` has no cells: it links it with `\pynotebook{}` and uses its variables and functions.

It can be opened and run in Jupyter or in VS Code like any notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Paths relative to the notebook's folder (Pyx runs it from here).
series = pd.read_csv('../data/annual_maxima.csv')
n_years = len(series)
mean = series['p_max_mm'].mean()
std = series['p_max_mm'].std()
series.describe()

## Method of moments fit

In [ ]:
alpha = np.sqrt(6) * std / np.pi          # scale parameter
u = mean - 0.5772 * alpha                  # location parameter

def quantile(T):
    """Maximum daily rainfall (mm) for a return period T (years)."""
    return u - alpha * np.log(-np.log(1 - 1 / T))

periods = [2, 5, 10, 25, 50, 100, 500]
quantiles = pd.DataFrame({'T': periods, 'P': [quantile(T) for T in periods]}).set_index('T')
quantiles

In [ ]:
def period_rows(periods):
    """Rows of a LaTeX table: period, annual probability and rainfall."""
    rows = []
    for T in periods:
        rows.append(f'{T} & {100 / T:.1f} & {quantile(T):.1f} \\\\')
    return '\n'.join(rows)

def intensity(T, duration_min):
    """Mean intensity (mm/h) for a duration, with a made-up IDF curve."""
    Id = quantile(T) / 24
    return Id * 11 ** ((28 ** 0.1 - (duration_min / 60) ** 0.1) / (28 ** 0.1 - 1))

print(period_rows([2, 10, 100]))

## Plot
Saved as an image in `figures/`; the document includes it with `\includegraphics`.

In [ ]:
ordered = np.sort(series['p_max_mm'].values)
F_emp = np.arange(1, n_years + 1) / (n_years + 1)
T_emp = 1 / (1 - F_emp)
T_curve = np.logspace(np.log10(1.05), np.log10(500), 200)
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.semilogx(T_curve, quantile(T_curve), color='#007acc', lw=2, label='Gumbel (moments)')
ax.semilogx(T_emp, ordered, 'o', ms=4, color='#cf222e', label='annual series')
ax.set_xlabel('return period T (years)')
ax.set_ylabel('max. daily rainfall (mm)')
ax.grid(alpha=0.3, which='both'); ax.legend(frameon=False)
fig.tight_layout()
import os
os.makedirs('figures', exist_ok=True)
fig.savefig('figures/gumbel.png', dpi=160)
trend = np.polyfit(series['year'], series['p_max_mm'], 1)[0]   # mm/year